# CatBoost

Цель: обучить CatBoost на пяти готовых фолдах и получить вероятности `satisfaction = 1`. Основная метрика — ROC AUC.

## Порядок работы

1. Загрузить `train_with_folds.parquet` и `test_clean.parquet`.
2. Отделить `satisfaction`, `fold` и `id` от признаков.
3. Оставить `Class` строкой и передать его через `cat_features=["Class"]`.
4. Создать массивы для OOF и усреднённых test-предсказаний.
5. Обучить пять моделей: текущий `fold` использовать для validation, остальные четыре — для обучения.
6. Использовать early stopping и считать ROC AUC каждого фолда.
7. Записать validation-вероятности в OOF, а test-вероятности усреднить по пяти моделям.

Начальные настройки: `iterations=2000`, `learning_rate=0.05`, `depth=8`, `eval_metric="AUC"`, `early_stopping_rounds=100`, `random_seed=42`.

Не создавать новый `train_test_split`, не масштабировать признаки и не заполнять числовые пропуски. Использовать `predict_proba(data)[:, 1]`, а не классы `0/1`.

## Результаты

Сохранить:

- `artifacts/oof/catboost.npy`;
- `artifacts/test_predictions/catboost.npy`;
- `artifacts/metrics/catboost.json` с параметрами, fold AUC, OOF AUC и лучшими итерациями.

Проверить длины массивов, диапазон вероятностей от 0 до 1 и отсутствие `id`, `fold`, `satisfaction` среди признаков.

In [22]:
from pathlib import Path
import json

import optuna
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

In [23]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
processed_data_dir = project_root / "data" / "processed"
raw_data_dir = project_root / "data" / "raw"
oof_dir = project_root / "artifacts" / "oof"
test_predictions_dir = project_root / "artifacts" / "test_predictions"
metrics_dir = project_root / "artifacts" / "metrics"

train_path = processed_data_dir / "train_with_folds.parquet"
test_path = processed_data_dir / "test_clean.parquet"
sample_submission_path = raw_data_dir / "sample_submission.csv"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)
sample_submission_df = pd.read_csv(sample_submission_path)

train_df.shape, test_df.shape, sample_submission_df.shape
train_df.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction,Arrival Delay Missing,fold
0,0,36,694,4,4,4,3,1,4,1,...,1,0,0.0,0,1,0,Eco,0,0,1
1,1,56,651,5,5,5,5,5,4,4,...,3,0,0.0,1,1,1,Business,1,0,3
2,2,31,1371,3,4,2,5,3,3,3,...,3,0,0.0,0,1,0,Eco,0,0,0
3,3,10,1616,3,5,3,3,3,3,3,...,3,0,0.0,1,1,0,Eco,0,0,3
4,4,23,481,3,4,3,4,5,3,5,...,5,5,0.0,0,0,1,Eco,0,0,4


In [24]:
target = train_df["satisfaction"]
fold_numbers = train_df["fold"]
train_ids = train_df["id"]
test_ids = test_df["id"]

features = train_df.drop(columns=["id", "satisfaction", "fold"])
test_features = test_df.drop(columns=["id"])

class_values = sorted(features["Class"].unique())
features["Class"] = pd.Categorical(features["Class"], categories=class_values)
test_features["Class"] = pd.Categorical(test_features["Class"], categories=class_values)

assert list(features.columns) == list(test_features.columns)

features.shape, test_features.shape, features["Class"].dtype

((699635, 22),
 (299844, 22),
 CategoricalDtype(categories=['Business', 'Eco', 'Eco Plus'], ordered=False, categories_dtype=str))

In [25]:
fold = 0
train_mask = fold_numbers != fold
validation_mask = fold_numbers == fold

x_train = features.loc[train_mask]
x_valid = features.loc[validation_mask]

y_train = target.loc[train_mask]
y_valid = target.loc[validation_mask]

x_train.shape, x_valid.shape, y_train.shape, y_valid.shape


((559708, 22), (139927, 22), (559708,), (139927,))

In [26]:
train_pool = Pool(
    data = x_train,
    label = y_train,
    cat_features = ["Class"]
)

valid_pool = Pool(
    data = x_valid,
    label = y_valid,
    cat_features = ["Class"]
)

train_pool.num_row(), train_pool.num_col(), valid_pool.num_row(), valid_pool.num_col(),

(559708, 22, 139927, 22)

In [27]:
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False

)

In [28]:
model.fit(
    train_pool,
    eval_set = valid_pool,
    early_stopping_rounds = 100,
    use_best_model = True
)

0:	test: 0.9459329	best: 0.9459329 (0)	total: 79.2ms	remaining: 1m 58s
100:	test: 0.9562812	best: 0.9562812 (100)	total: 7.49s	remaining: 1m 43s
200:	test: 0.9574412	best: 0.9574412 (200)	total: 15.2s	remaining: 1m 38s
300:	test: 0.9578611	best: 0.9578629 (299)	total: 24.4s	remaining: 1m 37s


KeyboardInterrupt: 

In [ ]:
valid_predictions = model.predict_proba(valid_pool)[:, 1]
valid_predictions

In [ ]:
auc = roc_auc_score(y_valid, valid_predictions)
print(auc, model.get_best_iteration())

In [ ]:
def objective(trial):
    depth = trial.suggest_int("depth", 6, 9)
    learning_rate = trial.suggest_float("learning_rate", 0.02, 0.1, log=True)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1, 20, log=True)
    random_strength = trial.suggest_float("random_strength", 0.1, 10, log=True)

    trial_model = CatBoostClassifier(
        iterations=2000,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    trial_model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=100,
        use_best_model=True,
    )

    return trial_model.get_best_score()["validation"]["AUC"]


In [ ]:
sampler = optuna.samplers.TPESampler(
    seed=42
)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)
sampler

осталось создать study с направлением maximize(меняет гиперпараметры чтобы максимизировать auc)
запустить на 8 пробниках
сохранить параметры
обучить на всех 5 фолдах
собрать предсказания, взять среднее
все сохранить


In [ ]:
study.optimize(objective, n_trials=8)

In [ ]:
print("Best result for AUC: ")
print(study.best_value)
print("Best parameters: ")
print(study.best_params)

In [ ]:
oof_predic = np.zeros(len(train_df))
test_predic = np.zeros(len(test_df))
fold_res = []
best_iter = []

test_pool = Pool(test_features, cat_features=["Class"])
for fold in range(5):
    print("Training fold number: ", fold)
    train_mask = fold_numbers != fold
    validation_mask = fold_numbers == fold

    x_train = features.loc[train_mask]
    y_train = target.loc[train_mask]

    x_valid = features.loc[validation_mask]
    y_valid = target.loc[validation_mask]

    train_pool = Pool(data=x_train, label=y_train, cat_features=["Class"])

    valid_pool = Pool(data=x_valid, label=y_valid, cat_features=["Class"])

    model = CatBoostClassifier(iterations=2000,
                               **study.best_params,
                               loss_function="Logloss",
                               eval_metric="AUC",
                               random_seed=42,
                               verbose=False,
                               allow_writing_files=False)

    model.fit(train_pool,
              eval_set=valid_pool,
              early_stopping_rounds=100,
              use_best_model=True)

    valid_predictions = model.predict_proba(valid_pool)[:, 1]
    oof_predic[validation_mask.to_numpy()] = valid_predictions

    fold_auc = roc_auc_score(y_valid, valid_predictions)
    fold_res.append(fold_auc)

    best_iter.append(model.get_best_iteration())

    test_predic += model.predict_proba(test_pool)[:, 1] / 5

    print("Each folds AUC: ", fold_auc)

In [ ]:
print("Fold AUCs: ", fold_res)
print("All OOF AUC: ", roc_auc_score(target, oof_predic))
print("All best iterations: ", best_iter)
print("OOF amount: ", oof_predic.shape)
print("Test amount: ", test_predic.shape)

In [ ]:
oof_dir.mkdir(parents=я True, exist_ok=True)
test_predictions_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

np.save(oof_dir/"catboost.npy", oof_predic)
np.save(test_predictions_dir/"catboost.npy", test_predic)

In [ ]:
all_metrics = {
    "model": "CatBoost",
    "oof_auc": float(roc_auc_score(target, oof_predic)),
    "fold_auc": [float(x) for x in fold_res],
    "best_iterations": [int(x) for x in best_iter],
    "best_params": study.best_params}

with open(metrics_dir/"catboost.json", "w") as f:
    json.dump(all_metrics, f, indent=4)